In [192]:
import pandas as pd
import numpy as np
from gurobipy import Model, GRB, quicksum

In [193]:
# Cell 2 (fixed): Load only the required columns
# ------------------------------------------------------------
# We explicitly select the columns as described. No assumptions.
# We're also going to rename them for consistent variable naming later.

buses = pd.read_csv("Buses.csv")[["bus", "demand"]]
branches = pd.read_csv("Branches.csv")[["branch", "from", "to", "y", "L"]]
generators = pd.read_csv("Generators.csv")[["generator", "bus", "fuel", "U", "sigma1", "sigma2", "X"]]

# Rename for convenience (optional but makes constraint code easier to read)
buses.rename(columns={"bus": "id", "demand": "Pd"}, inplace=True)
branches.rename(columns={"branch": "id", "from": "from_bus", "to": "to_bus"}, inplace=True)
generators.rename(columns={"generator": "id", "fuel": "type", "U": "Uh", "sigma1": "omega1", "sigma2": "omega2", "X": "Xh"}, inplace=True)

buses = buses.dropna()
branches = branches.dropna()
generators = generators.dropna()

buses["id"] = buses["id"].astype(int)
branches["from_bus"] = branches["from_bus"].astype(int)
branches["to_bus"] = branches["to_bus"].astype(int)
generators["id"] = generators["id"].astype(int)
generators["bus"] = generators["bus"].astype(int)

generators["Uh"] = generators["Uh"].clip(lower=0)
generators["Xh"] = generators["Xh"].clip(lower=0)

# Make sure Uh >= Xh always to avoid Pg_excess negative bounds
generators["Xh"] = generators[["Uh", "Xh"]].min(axis=1)


# Preview cleaned datasets
print("Buses (cleaned):\n", buses.head(), "\n")
print("Branches (cleaned):\n", branches.head(), "\n")
print("Generators (cleaned):\n", generators.head())

Buses (cleaned):
    id   Pd
0   0  0.0
1   1  0.0
2   2  0.0
3   3  0.0
4   4  0.0 

Branches (cleaned):
   id  from_bus  to_bus           y     L
0  0         0     335   13.082156  2.19
1  1         1      40   11.528706  2.84
2  2         2    1704   20.176341  7.94
3  3         3    1641  114.025086  4.76
4  4         4    1482    5.304758  2.19 

Generators (cleaned):
    id  bus     type     Uh   omega1   omega2        Xh
0   0   10  nuclear  12.99   412.00   412.00  4.330000
1   1   11  nuclear  10.12   412.00   412.00  3.373333
2   2   12      rfo   0.00  7095.64  7609.72  0.000000
3   3   12      rfo   0.00  7095.64  7609.72  0.000000
4   4   13      rfo   0.00  7024.00  7553.20  0.000000


In [194]:
# Cell 3 (fixed): Clean and validate datasets
# ------------------------------------------------------------
# Now we explicitly check for missing values and bad types.
# If anything looks off, we will flag it here.

# Check for missing values
print("Missing values check:")
print("Buses:", buses.isnull().sum().to_dict())
print("Branches:", branches.isnull().sum().to_dict())
print("Generators:", generators.isnull().sum().to_dict())

# Optional: Drop rows with missing critical data or set defaults (prof didn't specify, so we’ll warn instead)
if buses.isnull().any().any():
    raise ValueError("Missing values in Buses.csv! Please fix this before continuing.")
if branches.isnull().any().any():
    raise ValueError("Missing values in Branches.csv! Please fix this before continuing.")
if generators.isnull().any().any():
    raise ValueError("Missing values in Generators.csv! Please fix this before continuing.")

Missing values check:
Buses: {'id': 0, 'Pd': 0}
Branches: {'id': 0, 'from_bus': 0, 'to_bus': 0, 'y': 0, 'L': 0}
Generators: {'id': 0, 'bus': 0, 'type': 0, 'Uh': 0, 'omega1': 0, 'omega2': 0, 'Xh': 0}


In [195]:
# Cell 6: Define generator output and linearized cost components
# ------------------------------------------------------------
# We define:
# - Pg[h]: total power output of generator h
# - Pg_min[h]: min(Pg[h], Xh), costed at omega1
# - Pg_excess[h]: max(Pg[h] - Xh, 0), costed at omega2
# These let us represent the cost function linearly without PWL or binary variables.

Pg = {}           # Total power from generator h
Pg_min = {}       # Portion of Pg up to Xh
Pg_excess = {}    # Portion of Pg exceeding Xh

for _, row in generators.iterrows():
    h = int(row["id"])
    Uh = row["Uh"]
    Xh = row["Xh"]

    # Main generator output variable
    Pg[h] = model.addVar(lb=0, ub=Uh, name=f"Pg_{h}")

    # Cost modeling variables
    Pg_min[h] = model.addVar(lb=0, ub=min(Uh, Xh), name=f"Pg_min_{h}")
    Pg_excess[h] = model.addVar(lb=0, ub=max(0, Uh - Xh), name=f"Pg_excess_{h}")


model.update()

In [196]:
# Cell 7: Add linear constraints to connect Pg, Pg_min, Pg_excess
# ------------------------------------------------------------
# These constraints implement:
# - Pg_min <= Pg
# - Pg_min <= Xh (already handled by variable bound)
# - Pg_excess >= Pg - Xh
# - Pg_excess >= 0 (already handled by variable bound)

for _, row in generators.iterrows():
    h = int(row["id"])
    Xh = row["Xh"]

    model.addConstr(Pg_min[h] <= Pg[h], name=f"min_le_Pg_{h}")
    model.addConstr(Pg_excess[h] >= Pg[h] - Xh, name=f"excess_ge_Pg_minus_X_{h}")


In [197]:
# Cell 7: Define Sk (load shed), f (flow), and theta (angle) variables
# ------------------------------------------------------------
# Sk[k]: amount of unmet demand at bus k (we want this to be zero ideally)
# f[e]: power flow on branch e, constrained by line limit L
# theta[k]: voltage angle at bus k (we fix one bus to 0 later as a reference)

Sk = {}
f = {}
theta = {}

# Load shedding at each bus
for _, row in buses.iterrows():
    k = int(row["id"])
    Pd = row["Pd"]
    Sk[k] = model.addVar(lb=0, ub=max(0, Pd), name=f"Sk_{k}")

# Flow on each branch (line)
for _, row in branches.iterrows():
    e = int(row["id"])
    L = row["L"]
    f[e] = model.addVar(lb=-L, ub=L, name=f"f_{e}")  # symmetric flow limits

# Voltage angles at buses
for k in buses["id"]:
    theta[int(k)] = model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"theta_{int(k)}")

model.update()


In [198]:
# Cell 8: Add line flow constraints and fix the reference bus angle
# ------------------------------------------------------------
# For each line: f[e] = y * (theta[from] - theta[to])
# Also, fix the angle of the first bus (id = 0) to be 0, which is standard in power flow models.

for _, row in branches.iterrows():
    e = int(row["id"])
    i = int(row["from_bus"])
    j = int(row["to_bus"])
    y = row["y"]

    model.addConstr(f[e] == y * (theta[i] - theta[j]), name=f"flow_eq_{e}")

# Fix reference bus angle
ref_bus_id = int(buses["id"].iloc[0])  # first bus
model.addConstr(theta[ref_bus_id] == 0, name="ref_bus")


<gurobi.Constr *Awaiting Model Update*>

In [199]:
# Precompute inflow and outflow branches for each bus
from_map = {}  # bus_id -> list of branches where it's the from_bus
to_map = {}    # bus_id -> list of branches where it's the to_bus

for _, row in branches.iterrows():
    e = int(row["id"])
    i = int(row["from_bus"])
    j = int(row["to_bus"])

    from_map.setdefault(i, []).append(e)
    to_map.setdefault(j, []).append(e)


In [200]:
# Cell 9 (fixed): Add flow conservation constraints efficiently
# ------------------------------------------------------------

for _, row in buses.iterrows():
    k = int(row["id"])
    Pd = row["Pd"]

    inflow = to_map.get(k, [])
    outflow = from_map.get(k, [])

    gens_at_bus = generators[generators["bus"] == k]["id"].astype(int).tolist()

    lhs = quicksum(f[e] for e in inflow) - quicksum(f[e] for e in outflow)
    rhs = quicksum(Pg[h] for h in gens_at_bus) - (Pd - Sk[k])

    model.addConstr(lhs == rhs, name=f"flow_balance_bus_{k}")


In [201]:
gen_cost = quicksum(
    row["omega1"] * Pg_min[int(row["id"])] + row["omega2"] * Pg_excess[int(row["id"])]
    for _, row in generators.iterrows()
)

shed_cost = quicksum(Sk[k] for k in Sk)

model.setObjective(gen_cost + 1e6 * shed_cost, GRB.MINIMIZE)

In [202]:
model.optimize()

if model.status == GRB.INFEASIBLE:
    print("❌ Model is infeasible. Computing IIS (infeasibility certificate)...")
    model.computeIIS()
    for var in model.getVars():
        if var.IISLB or var.IISUB:
            print(f"❗ Infeasible bound on variable: {var.VarName}")
            print(f"   Lower Bound = {var.LB}, Upper Bound = {var.UB}")

    model.write("model.ilp")  # You can inspect this in a text editor

    # Print a few of the constraints that are causing problems
    for c in model.getConstrs():
        if c.IISConstr:
            print(f"Infeasible constraint: {c.ConstrName}")

if model.status == GRB.OPTIMAL:
    print("✅ Optimal solution found!")
    total_shed = sum(Sk[k].X for k in Sk)
    print(f"Total demand shed: {total_shed:.4f}")
else:
    print("❌ No optimal solution. Status:", model.status)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D60)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 45051 rows, 70555 columns and 139706 nonzeros
Model fingerprint: 0x3c99ba51
Variable types: 70160 continuous, 395 integer (395 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+05]
  Objective range  [4e+02, 1e+06]
  Bounds range     [2e-04, 2e+01]
  RHS range        [2e-04, 5e+00]
Presolve removed 0 rows and 9 columns
Presolve time: 0.01s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 1 (of 10 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
❌ Model is infeasible. Computing IIS (infeasibility certificate)...
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D60)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, us